combined PAC analysis. joins the temporal classifier (nb 05, all_bylines_bands.csv) and the content classifier (nb 06, content_classifier_per_advertiser.csv) per-advertiser, computes a combined PAC-likeness score, and produces two tables for the report:

1. top 20 advertisers by total spend, with their three metrics (band / EC index, mean election_prob inside campaign, content shift) and the combined PAC score.
2. top 10 PACs (highest combined score), with their top 3 ads by spend each.

the combined score is `pac_score = ec_index * mean_prob_inside` --- both metrics are bounded [0, 1], so the product is a strict-AND composite: high only if temporal concentration AND election-content density are high together.

setup.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, expr, coalesce, substring, first,
    sum as spark_sum, count as spark_count,
    row_number, desc,
)
from pyspark.sql.window import Window
import pandas as pd
import numpy as np

spark = SparkSession.builder \
    .appName('FB_API_combined_pacs') \
    .config('spark.sql.parquet.output.committer.class',
            'org.apache.parquet.hadoop.ParquetOutputCommitter') \
    .config('mapreduce.fileoutputcommitter.algorithm.version', '2') \
    .getOrCreate()

print('Master:', spark.sparkContext.master)
print('Spark version:', spark.version)

paths.

In [ ]:
V3_PATH        = '/user/s3348393/main/preprocessing/v3/parquet'
BANDS_CSV      = '../data/all_bylines_bands.csv'
CONTENT_CSV    = '../data/content_classifier_per_advertiser.csv'

TOP_20_OUT_CSV     = '../data/top_20_with_metrics.csv'
TOP_PACS_OUT_CSV   = '../data/top_10_pacs_with_top_ads.csv'

TOP_N_SPENDERS    = 20
TOP_N_PACS        = 10
TOP_N_ADS_PER_PAC = 3

load temporal and content scores per advertiser, join them, compute the combined PAC score.

In [ ]:
bands_df   = pd.read_csv(BANDS_CSV)
content_df = pd.read_csv(CONTENT_CSV)

print(f'temporal classifier rows: {len(bands_df):,}')
print(f'content classifier rows:  {len(content_df):,}')

# Inner-join on bylines --- only advertisers that survived both classifiers
metrics = (bands_df[['bylines', 'band', 'persistence', 'ec_index']]
    .merge(
        content_df[['bylines', 'mean_prob_inside', 'shift',
                    'mean_election_prob_weighted', 'total_spend']],
        on='bylines', how='inner'
    ))

# Combined PAC-likeness score: strict-AND multiplicative composite
metrics['pac_score'] = metrics['ec_index'] * metrics['mean_prob_inside']

print(f'\njoined metrics: {len(metrics):,} advertisers')
print(f'pac_score range: {metrics["pac_score"].min():.3f} - {metrics["pac_score"].max():.3f}')

table 1: top 20 advertisers by total spend, with their three metrics and the combined PAC score.

In [ ]:
top20_metrics = (metrics
    .sort_values('total_spend', ascending=False)
    .head(TOP_N_SPENDERS)
    .reset_index(drop=True))

display_cols = ['bylines', 'total_spend', 'band',
                'ec_index', 'mean_prob_inside', 'shift', 'pac_score']

print(f'Top {TOP_N_SPENDERS} advertisers by total spend:')
display(top20_metrics[display_cols])

top20_metrics[display_cols].to_csv(TOP_20_OUT_CSV, index=False)
print(f'\nWrote {TOP_20_OUT_CSV}')

table 2: top 10 PACs by combined score, with each PAC's top 3 ads by spend.

In [ ]:
# Identify the top 10 PACs by combined PAC score
top_pacs = (metrics
    .sort_values('pac_score', ascending=False)
    .head(TOP_N_PACS)
    .reset_index(drop=True))
top_pac_bylines = top_pacs['bylines'].tolist()

print(f'Top {TOP_N_PACS} PACs by pac_score:')
display(top_pacs[['bylines', 'pac_score', 'band', 'ec_index',
                  'mean_prob_inside', 'shift', 'total_spend']])

# Pull each PAC's top 3 ads from v3 (by total spend per unique creative title)
def first_non_empty(col_name):
    return expr(f"filter({col_name}, x -> x is not null and length(x) > 0)[0]")

ad_title = coalesce(
    first_non_empty('creative_link_titles'),
    substring(first_non_empty('creative_bodies'), 1, 80),
)

v3 = spark.read.parquet(V3_PATH).filter(col('match_type').isNull())

ad_spend = (v3
    .filter(col('bylines').isin(top_pac_bylines) & col('spend_mid').isNotNull())
    .withColumn('ad_title', ad_title)
    .filter(col('ad_title').isNotNull())
    .groupBy('bylines', 'ad_title')
    .agg(
        spark_sum('spend_mid').alias('total_spend'),
        spark_count('*').alias('n_runs'),
        first('ad_snapshot_url').alias('example_url'),
    ))

w = Window.partitionBy('bylines').orderBy(desc('total_spend'))
top_pac_ads = (ad_spend
    .withColumn('rank', row_number().over(w))
    .filter(col('rank') <= TOP_N_ADS_PER_PAC)
    .orderBy('bylines', 'rank')
    .toPandas())

# Preserve PAC ranking order
top_pac_ads['_byline_order'] = top_pac_ads['bylines'].map(
    {b: i for i, b in enumerate(top_pac_bylines)})
top_pac_ads = (top_pac_ads
    .sort_values(['_byline_order', 'rank'])
    .drop(columns='_byline_order')
    .reset_index(drop=True))

print(f'\n{len(top_pac_ads)} rows: top {TOP_N_ADS_PER_PAC} ads for each of the top {TOP_N_PACS} PACs.')

per-PAC display and export.

In [ ]:
for byline in top_pac_bylines:
    sub = top_pac_ads[top_pac_ads['bylines'] == byline]
    pac_row = top_pacs[top_pacs['bylines'] == byline].iloc[0]
    print(f'\n=== {byline}  (pac_score={pac_row["pac_score"]:.3f}, '
          f'${pac_row["total_spend"]:,.0f} total, band={pac_row["band"]}) ===')
    if sub.empty:
        print('(no ads matched filter)')
        continue
    display(sub[['rank', 'ad_title', 'total_spend', 'n_runs']])

# Join PAC-level scores onto the ad-level rows for the export
top_pac_ads_export = top_pac_ads.merge(
    top_pacs[['bylines', 'pac_score', 'band']],
    on='bylines', how='left')

out_cols = ['bylines', 'pac_score', 'band', 'rank', 'ad_title',
            'total_spend', 'n_runs', 'example_url']
top_pac_ads_export[out_cols].to_csv(TOP_PACS_OUT_CSV, index=False)
print(f'\nWrote {TOP_PACS_OUT_CSV}')